# Notebook 07e — Phase A: Strict Mondrian Conformal Protocol

**Problem (TA / §10.4.1 catch):** v2 Mondrian thresholds were fitted on `test_set \ canonical_1000`, which is still test data. Strict conformal requires a holdout disjoint from training, calibrator fitting, AND evaluation.

**Phase A fix (Option β):** Carve `X_calib` into two disjoint slices:
- `X_calib_strict` (~80%) → refit per-class hybrid calibrators
- `X_mondrian_holdout` (~20%) → fit Mondrian thresholds

Rare classes (n<25 in y_calib) stay 100% in strict slice (NSL U2R n=10, CIC U2R n=7) — they fall back to marginal threshold under Mondrian anyway.

**Models stay frozen.** v2 macro-F1 numbers are not affected.

**All outputs use `*_strict.*` suffix.** v2 files untouched.


In [1]:
# Setup: mount drive, copy git creds
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'Ready in: {os.getcwd()}')

Mounted at /content/drive
Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
# Imports and constants
import numpy as np
import pandas as pd
import json, time, joblib
from pathlib import Path
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

SEED = 42
np.random.seed(SEED)

DATASETS = ['nsl_kdd_v2']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
VARIANTS = ['5class_cw', '5class_smote']
MODELS_PER_DATASET = [f'{a}_{v}' for v in VARIANTS for a in ARCHITECTURES]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

# Phase A split
HOLDOUT_FRAC = 0.20
RARE_CLASS_THRESHOLD = 25

# Identical to v2 for direct comparability
PLATT_THRESHOLD = 30
MIN_CALIB_MONDRIAN = 30

# Conformal
ALPHAS = [0.05, 0.10, 0.20]
ALPHA_PRIMARY = 0.05

# Health flag (identical to 07d)
T_GREEN_LO, T_GREEN_HI = 0.05, 0.95
T_RED_HI, T_RED_LO = 0.999, 0.001
CLIFF_GREEN_HI, CLIFF_RED_LO = 0.05, 0.20
CLIFF_THRESH = 0.95
N_GREEN_LO, N_RED_HI = 100, 30

# Bootstrap
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42

EPS = 1e-6

print(f'Scope: {len(DATASETS)} datasets x {len(MODELS_PER_DATASET)} models = {len(DATASETS) * len(MODELS_PER_DATASET)} cells')

Scope: 1 datasets x 6 models = 6 cells


In [3]:
# Path resolution helper (NSL uses predictions/, UNSW+CIC use probabilities/)
def find_proba_file(dataset, model_name, split):
    fname = f'{model_name}_{split}_proba.npy'
    for subdir in ['probabilities', 'predictions']:
        p = Path(REPO) / 'models' / dataset / subdir / fname
        if p.exists():
            return p
    raise FileNotFoundError(f'No {fname} for {dataset}/{model_name}')

for ds in DATASETS:
    p = find_proba_file(ds, 'rf_5class_cw', 'calib')
    print(f'{ds}: {p.relative_to(REPO)}')

nsl_kdd_v2: models/nsl_kdd_v2/predictions/rf_5class_cw_calib_proba.npy


In [4]:
# Class-aware 80/20 split of X_calib
# Rare classes (n<25) stay 100% in strict slice
split_indices = {}

for ds in DATASETS:
    y_calib = np.load(Path(REPO) / 'data' / 'processed' / ds / 'y_calib_5class.npy')
    n = len(y_calib)
    rng = np.random.RandomState(SEED)

    class_counts = Counter(y_calib.tolist())
    rare = [c for c, cnt in class_counts.items() if cnt < RARE_CLASS_THRESHOLD]
    common = [c for c, cnt in class_counts.items() if cnt >= RARE_CLASS_THRESHOLD]

    strict_mask = np.zeros(n, dtype=bool)
    for c in rare:
        strict_mask[y_calib == c] = True
    for c in common:
        idx_c = np.where(y_calib == c)[0]
        rng.shuffle(idx_c)
        n_strict = int(round(len(idx_c) * (1 - HOLDOUT_FRAC)))
        strict_mask[idx_c[:n_strict]] = True

    strict_idx = np.where(strict_mask)[0]
    holdout_idx = np.where(~strict_mask)[0]

    split_indices[ds] = {
        'strict_idx': strict_idx,
        'holdout_idx': holdout_idx,
        'rare_classes_handled': rare,
    }

    print(f'\n=== {ds} ===')
    print(f'  Total: {n} | Strict: {len(strict_idx)} | Holdout: {len(holdout_idx)}')
    print(f'  Rare classes all-in-strict: {[CLASS_NAMES_5[c] for c in rare]} (counts: {[class_counts[c] for c in rare]})')
    print(f'  Per-class distribution:')
    for c in range(5):
        s = int((y_calib[strict_idx] == c).sum())
        h = int((y_calib[holdout_idx] == c).sum())
        print(f'    {CLASS_NAMES_5[c]:8s}: strict={s:>6d}, holdout={h:>6d}')


=== nsl_kdd_v2 ===
  Total: 25195 | Strict: 20158 | Holdout: 5037
  Rare classes all-in-strict: ['U2R'] (counts: [10])
  Per-class distribution:
    Normal  : strict= 10775, holdout=  2694
    DoS     : strict=  7349, holdout=  1837
    Probe   : strict=  1865, holdout=   466
    R2L     : strict=   159, holdout=    40
    U2R     : strict=    10, holdout=     0


In [5]:
# Persist split indices for reproducibility
for ds in DATASETS:
    out_dir = Path(REPO) / 'calibrators' / ds
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / 'X_calib_strict_indices.npy', split_indices[ds]['strict_idx'])
    np.save(out_dir / 'X_mondrian_holdout_indices.npy', split_indices[ds]['holdout_idx'])
    print(f'{ds}: split indices saved')

nsl_kdd_v2: split indices saved


In [6]:
# Hybrid calibrator refit (identical logic to Notebook 03e)
def fit_calibrator(p_calib, y_indicator, n_class):
    if n_class >= PLATT_THRESHOLD:
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(p_calib, y_indicator)
        return cal, 'isotonic'
    else:
        cal = LogisticRegression(C=1e10, solver='lbfgs')
        cal.fit(p_calib.reshape(-1, 1), y_indicator)
        return cal, 'platt'

def apply_calibrator(calibrator, strategy, p_test):
    if strategy == 'isotonic':
        return calibrator.predict(p_test)
    return calibrator.predict_proba(p_test.reshape(-1, 1))[:, 1]

def refit_and_apply_strict(dataset, model_name):
    proc = Path(REPO) / 'data' / 'processed' / dataset
    y_calib_full = np.load(proc / 'y_calib_5class.npy')
    p_calib_2d_full = np.load(find_proba_file(dataset, model_name, 'calib'))
    p_test_2d = np.load(find_proba_file(dataset, model_name, 'test'))

    strict_idx = split_indices[dataset]['strict_idx']
    holdout_idx = split_indices[dataset]['holdout_idx']
    y_calib_strict = y_calib_full[strict_idx]
    p_calib_2d_strict = p_calib_2d_full[strict_idx]

    n_classes = p_calib_2d_full.shape[1]
    calib_counts = Counter(y_calib_strict.tolist())

    calibrators, strategies = {}, {}
    p_test_cal_strict = np.zeros_like(p_test_2d)

    for c in range(n_classes):
        y_c = (y_calib_strict == c).astype(int)
        p_c = p_calib_2d_strict[:, c]
        n_c = calib_counts.get(c, 0)
        cal, strat = fit_calibrator(p_c, y_c, n_c)
        calibrators[c] = cal
        strategies[c] = strat
        p_test_cal_strict[:, c] = apply_calibrator(cal, strat, p_test_2d[:, c])

    p_holdout_2d = p_calib_2d_full[holdout_idx]
    p_holdout_cal = np.zeros_like(p_holdout_2d)
    for c in range(n_classes):
        p_holdout_cal[:, c] = apply_calibrator(calibrators[c], strategies[c], p_holdout_2d[:, c])

    for arr in [p_test_cal_strict, p_holdout_cal]:
        row_sums = arr.sum(axis=1, keepdims=True)
        row_sums = np.where(row_sums == 0, 1, row_sums)
        arr /= row_sums

    return {
        'calibrators': calibrators, 'strategies': strategies,
        'calib_counts': dict(calib_counts), 'n_classes': n_classes,
        'p_test_cal_strict': p_test_cal_strict,
        'p_holdout_cal': p_holdout_cal,
    }

print('Calibrator helpers ready')

Calibrator helpers ready


In [7]:
# Refit all 18 hybrid calibrators on strict slice
print('=' * 70)
print('STRICT CALIBRATOR REFIT')
print('=' * 70)

strict_calibrated_probs = {}
strict_holdout_probs = {}

t0 = time.time()
for ds in DATASETS:
    print(f'\n--- {ds} ---')
    for model_name in MODELS_PER_DATASET:
        try:
            res = refit_and_apply_strict(ds, model_name)
            strict_calibrated_probs[(ds, model_name)] = res['p_test_cal_strict']
            strict_holdout_probs[(ds, model_name)] = res['p_holdout_cal']

            out_dir = Path(REPO) / 'calibrators' / ds
            np.save(out_dir / f'{model_name}_test_proba_strict.npy', res['p_test_cal_strict'])
            np.save(out_dir / f'{model_name}_holdout_proba_strict.npy', res['p_holdout_cal'])
            joblib.dump({
                'calibrators': res['calibrators'], 'strategies': res['strategies'],
                'calib_counts': res['calib_counts'], 'n_classes': res['n_classes'],
                'platt_threshold': PLATT_THRESHOLD,
            }, out_dir / f'{model_name}_hybrid_strict.joblib')

            strat_str = ','.join(res['strategies'][c][0] for c in range(5))
            print(f'  {model_name:<22} strategies=[{strat_str}] (i=iso, p=platt)')
        except Exception as e:
            print(f'  {model_name:<22} ERROR: {type(e).__name__}: {e}')

print(f'\nRefit complete in {(time.time()-t0)/60:.1f} min, cached {len(strict_calibrated_probs)} cells')

STRICT CALIBRATOR REFIT

--- nsl_kdd_v2 ---
  rf_5class_cw           strategies=[i,i,i,i,p] (i=iso, p=platt)
  xgb_5class_cw          strategies=[i,i,i,i,p] (i=iso, p=platt)
  dnn_5class_cw          strategies=[i,i,i,i,p] (i=iso, p=platt)
  rf_5class_smote        strategies=[i,i,i,i,p] (i=iso, p=platt)
  xgb_5class_smote       strategies=[i,i,i,i,p] (i=iso, p=platt)
  dnn_5class_smote       strategies=[i,i,i,i,p] (i=iso, p=platt)

Refit complete in 0.2 min, cached 6 cells


In [8]:
# Conformal helpers (identical math to 07c)
def split_conformal_threshold(probs, y_true, alpha):
    n = len(y_true)
    scores = 1.0 - probs[np.arange(n), y_true]
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(scores, q_level))

def mondrian_conformal_thresholds(probs, y_true, alpha, n_classes=5, min_calib=30):
    y_pred = probs.argmax(axis=1)
    marginal = split_conformal_threshold(probs, y_true, alpha)
    thresholds, fallback, n_per_class = {}, [], {}
    for c in range(n_classes):
        mask = (y_pred == c)
        n_c = int(mask.sum())
        n_per_class[c] = n_c
        if n_c < min_calib:
            thresholds[c] = marginal
            fallback.append(c)
        else:
            scores_c = 1.0 - probs[mask, :][np.arange(n_c), y_true[mask]]
            q = min(np.ceil((n_c + 1) * (1 - alpha)) / n_c, 1.0)
            thresholds[c] = float(np.quantile(scores_c, q))
    return thresholds, fallback, n_per_class

def empirical_coverage_mondrian(probs, y_true, thresholds):
    y_pred = probs.argmax(axis=1)
    scores = 1.0 - probs[np.arange(len(y_true)), y_true]
    sample_t = np.array([thresholds[int(p)] for p in y_pred])
    return float((scores <= sample_t).mean())

def component_3_mondrian(probs, y_pred, thresholds):
    sample_t = np.array([thresholds[int(p)] for p in y_pred], dtype=np.float32)
    s = 1.0 - probs[np.arange(len(y_pred)), y_pred]
    return np.clip(1.0 - s / sample_t, 0.0, 1.0).astype(np.float32)

print('Conformal helpers loaded')

Conformal helpers loaded


In [9]:
# Load c2 stability (UNCHANGED from v2 — stability doesn't depend on calibrator)
stab_path = Path(REPO) / 'results' / 'tables' / 'stability_v2_per_sample_jaccard.csv'
df_stab = pd.read_csv(stab_path)
worst = df_stab.groupby(['dataset', 'model', 'sample_position'])['jaccard_top10'].min().reset_index()
worst.rename(columns={'jaccard_top10': 'worst_jaccard'}, inplace=True)

c2_lookup = {}
for (ds, m), g in worst.groupby(['dataset', 'model']):
    c2_lookup[(ds, m)] = g.sort_values('sample_position')['worst_jaccard'].values.astype(np.float32)

print(f'c2 lookup loaded for {len(c2_lookup)} cells (unchanged from v2)')

c2 lookup loaded for 6 cells (unchanged from v2)


In [10]:
# Strict SCTS computation on canonical 1000
print('=' * 70)
print('STRICT SCTS-v2 COMPUTATION ON CANONICAL 1000')
print('=' * 70)

scts_records, alpha_records = [], []
conformal_meta_strict = {}
validation_records, per_class_records = [], []

t0 = time.time()
for ds in DATASETS:
    print(f'\n--- {ds} ---')
    canonical_eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    y_test_full = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    y_canonical = y_test_full[canonical_eval_idx]

    holdout_idx = split_indices[ds]['holdout_idx']
    y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
    y_mondrian_holdout = y_calib_full[holdout_idx]

    for model_name in MODELS_PER_DATASET:
        p_test_strict = strict_calibrated_probs[(ds, model_name)]
        p_holdout_strict = strict_holdout_probs[(ds, model_name)]
        p_canonical = p_test_strict[canonical_eval_idx]
        y_pred_canonical = p_canonical.argmax(axis=1)

        c1 = p_canonical[np.arange(len(y_pred_canonical)), y_pred_canonical].astype(np.float32)
        c2 = c2_lookup[(ds, model_name)]

        # KEY: fit Mondrian on STRICT HOLDOUT (the fix)
        mthresh, fb_classes, n_per_c = mondrian_conformal_thresholds(
            p_holdout_strict, y_mondrian_holdout, ALPHA_PRIMARY, 5, MIN_CALIB_MONDRIAN
        )
        marg = split_conformal_threshold(p_holdout_strict, y_mondrian_holdout, ALPHA_PRIMARY)
        c3 = component_3_mondrian(p_canonical, y_pred_canonical, mthresh)
        emp_cov = empirical_coverage_mondrian(p_canonical, y_canonical, mthresh)

        geo = (np.clip(c1, EPS, 1) * np.clip(c2, EPS, 1) * np.clip(c3, EPS, 1)) ** (1/3)
        scts = (geo * 100).astype(np.float32)
        correct = (y_pred_canonical == y_canonical).astype(float)

        for i in range(len(scts)):
            scts_records.append({
                'dataset': ds, 'model': model_name, 'sample_position': i,
                'true_class': int(y_canonical[i]), 'pred_class': int(y_pred_canonical[i]),
                'correct': int(correct[i]),
                'c1': float(c1[i]), 'c2': float(c2[i]), 'c3': float(c3[i]),
                'scts': float(scts[i]),
            })

        conformal_meta_strict[f'{ds}/{model_name}'] = {
            'marginal_threshold_alpha_0.05': float(marg),
            'mondrian_thresholds_alpha_0.05': {str(c): float(t) for c, t in mthresh.items()},
            'fallback_classes': [int(c) for c in fb_classes],
            'n_calib_per_predicted_class': {str(c): int(n) for c, n in n_per_c.items()},
            'empirical_coverage_on_canonical_mondrian': float(emp_cov),
            'n_mondrian_holdout_samples': int(len(holdout_idx)),
        }

        for alpha in ALPHAS:
            m_a, fb_a, _ = mondrian_conformal_thresholds(p_holdout_strict, y_mondrian_holdout, alpha, 5, MIN_CALIB_MONDRIAN)
            marg_a = split_conformal_threshold(p_holdout_strict, y_mondrian_holdout, alpha)
            c3_a = component_3_mondrian(p_canonical, y_pred_canonical, m_a)
            geo_a = (np.clip(c1, EPS, 1) * np.clip(c2, EPS, 1) * np.clip(c3_a, EPS, 1)) ** (1/3)
            scts_a = geo_a * 100
            ec_a = empirical_coverage_mondrian(p_canonical, y_canonical, m_a)
            alpha_records.append({
                'dataset': ds, 'model': model_name, 'alpha': alpha,
                'mean_threshold_across_classes': float(np.mean(list(m_a.values()))),
                'marginal_threshold_reference': float(marg_a),
                'empirical_coverage_mondrian': float(ec_a),
                'n_fallback_classes': len(fb_a),
                'mean_scts': float(scts_a.mean()), 'median_scts': float(np.median(scts_a)),
            })

        overall_acc = correct.mean()
        pearson = np.corrcoef(scts, correct)[0, 1] if scts.std() > 1e-9 else 0.0
        for lo, hi in [(0, 25), (25, 50), (50, 75), (75, 101)]:
            mask = (scts >= lo) & (scts < hi)
            n = int(mask.sum())
            validation_records.append({
                'dataset': ds, 'model': model_name,
                'scts_bin_low': lo, 'scts_bin_high': hi, 'n': n,
                'accuracy': float(correct[mask].mean()) if n > 0 else float('nan'),
                'overall_accuracy': float(overall_acc),
                'pearson_corr_scts_correct': float(pearson),
            })

        for c, cname in enumerate(CLASS_NAMES_5):
            m = y_canonical == c
            if m.sum() > 0:
                per_class_records.append({
                    'dataset': ds, 'model': model_name,
                    'true_class': cname, 'class_idx': c, 'n': int(m.sum()),
                    'mean_scts': float(scts[m].mean()),
                    'mean_c1': float(c1[m].mean()), 'mean_c2': float(c2[m].mean()), 'mean_c3': float(c3[m].mean()),
                    'accuracy': float(correct[m].mean()),
                })

        print(f'  {model_name:<22} marg={marg:.3f} mondrian=[{",".join(f"{mthresh[c]:.2f}" for c in range(5))}] '
              f'fb={len(fb_classes)}/5 cov={emp_cov:.3f} meanSCTS={scts.mean():.1f} acc={overall_acc:.3f} r={pearson:+.3f}')

print(f'\nStrict SCTS complete in {(time.time()-t0)/60:.1f} min')

STRICT SCTS-v2 COMPUTATION ON CANONICAL 1000

--- nsl_kdd_v2 ---
  rf_5class_cw           marg=0.000 mondrian=[0.00,0.00,0.00,0.61,0.00] fb=1/5 cov=0.542 meanSCTS=1.8 acc=0.637 r=+0.013
  xgb_5class_cw          marg=0.000 mondrian=[0.00,0.00,0.00,0.43,0.00] fb=1/5 cov=0.555 meanSCTS=9.9 acc=0.636 r=+0.120
  dnn_5class_cw          marg=0.022 mondrian=[0.03,0.00,0.18,0.86,0.02] fb=1/5 cov=0.555 meanSCTS=46.8 acc=0.628 r=-0.008
  rf_5class_smote        marg=0.000 mondrian=[0.00,0.00,0.00,0.71,0.00] fb=1/5 cov=0.511 meanSCTS=1.5 acc=0.611 r=-0.030
  xgb_5class_smote       marg=0.000 mondrian=[0.00,0.00,0.00,0.03,0.00] fb=1/5 cov=0.593 meanSCTS=2.6 acc=0.638 r=+0.106
  dnn_5class_smote       marg=0.004 mondrian=[0.00,0.00,0.01,0.93,0.00] fb=1/5 cov=0.529 meanSCTS=15.8 acc=0.623 r=+0.199

Strict SCTS complete in 0.0 min


In [11]:
# Save strict CSVs and summary JSON
df_scts_strict = pd.DataFrame(scts_records)
df_perclass_strict = pd.DataFrame(per_class_records)
df_alpha_strict = pd.DataFrame(alpha_records)
df_val_strict = pd.DataFrame(validation_records)

out_dir = Path(REPO) / 'results' / 'tables'
df_scts_strict.to_csv(out_dir / 'scts_v2_canonical_strict.csv', index=False)
df_perclass_strict.to_csv(out_dir / 'scts_v2_per_class_summary_strict.csv', index=False)
df_alpha_strict.to_csv(out_dir / 'scts_v2_alpha_sensitivity_strict.csv', index=False)
df_val_strict.to_csv(out_dir / 'scts_v2_validation_strict.csv', index=False)

df_scts_strict['architecture'] = df_scts_strict['model'].apply(
    lambda m: 'rf' if 'rf' in m else ('xgb' if 'xgb' in m else 'dnn')
)

def to_json_safe(obj):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if isinstance(k, tuple):
                k = '|'.join(str(x) for x in k)
            elif not isinstance(k, (str, int, float, bool)) and k is not None:
                k = str(k)
            out[k] = to_json_safe(v)
        return out
    if isinstance(obj, list):
        return [to_json_safe(x) for x in obj]
    if isinstance(obj, (np.bool_, np.generic)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

summary_strict = {
    'timestamp': datetime.now().isoformat(),
    'protocol': 'phase_a_strict_holdout_mondrian',
    'mondrian_holdout_source': 'X_calib 20% class-aware slice (rare<25 stay 100% in strict)',
    'calibrator_source': 'X_calib 80% slice (isotonic n>=30, Platt n<30)',
    'n_datasets': len(DATASETS),
    'n_models': len(MODELS_PER_DATASET) * len(DATASETS),
    'n_samples_per_model': 1000,
    'primary_alpha': ALPHA_PRIMARY, 'sensitivity_alphas': ALPHAS,
    'overall_stats': {
        'mean_scts': float(df_scts_strict['scts'].mean()),
        'median_scts': float(df_scts_strict['scts'].median()),
        'min_scts': float(df_scts_strict['scts'].min()),
        'max_scts': float(df_scts_strict['scts'].max()),
        'std_scts': float(df_scts_strict['scts'].std()),
    },
    'mean_components': {
        'c1_calibration': float(df_scts_strict['c1'].mean()),
        'c2_stability': float(df_scts_strict['c2'].mean()),
        'c3_conformal': float(df_scts_strict['c3'].mean()),
    },
    'mean_scts_by_dataset_architecture': df_scts_strict.groupby(['dataset', 'architecture'])['scts'].mean().to_dict(),
    'mean_pearson_corr_scts_correctness': float(
        df_val_strict.groupby(['dataset', 'model'])['pearson_corr_scts_correct'].first().mean()
    ),
    'conformal_thresholds': conformal_meta_strict,
}

with open(out_dir / 'scts_v2_summary_strict.json', 'w') as f:
    json.dump(to_json_safe(summary_strict), f, indent=2)

print('Saved 5 files: 4 CSVs + summary JSON')
print(f'  Overall mean SCTS (strict): {summary_strict["overall_stats"]["mean_scts"]:.2f}')
print(f'  Overall mean c3 (strict): {summary_strict["mean_components"]["c3_conformal"]:.3f}')

Saved 5 files: 4 CSVs + summary JSON
  Overall mean SCTS (strict): 13.07
  Overall mean c3 (strict): 0.125


In [12]:
# Strict health flag — s1 uses new thresholds, s2 cliffs from holdout, s3 from strict
print('=' * 70)
print('STRICT HEALTH FLAG')
print('=' * 70)

cliff_data_strict = {}
for ds in DATASETS:
    holdout_idx = split_indices[ds]['holdout_idx']
    y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
    y_holdout = y_calib_full[holdout_idx]
    for model_name in MODELS_PER_DATASET:
        p_holdout = strict_holdout_probs[(ds, model_name)]
        scores = 1.0 - p_holdout[np.arange(len(y_holdout)), y_holdout]
        y_pred_h = p_holdout.argmax(axis=1)
        for pred_cls in range(5):
            mask = y_pred_h == pred_cls
            n_in = int(mask.sum())
            cliff_frac = float('nan') if n_in == 0 else float((scores[mask] >= CLIFF_THRESH).mean())
            cliff_data_strict[(ds, model_name, pred_cls)] = {
                'cliff_fraction': cliff_frac, 'n_pred_class_in_holdout': n_in,
            }

def flag_threshold(t):
    if t >= T_RED_HI or t < T_RED_LO: return 'red'
    if t >= T_GREEN_HI or t <= T_GREEN_LO: return 'amber'
    return 'green'

def flag_cliff(f):
    if np.isnan(f): return 'red'
    if f >= CLIFF_RED_LO: return 'red'
    if f >= CLIFF_GREEN_HI: return 'amber'
    return 'green'

def flag_support(n):
    if n < N_RED_HI: return 'red'
    if n < N_GREEN_LO: return 'amber'
    return 'green'

def combine_flags(*flags):
    if 'red' in flags: return 'red'
    if 'amber' in flags: return 'amber'
    return 'green'

health_records_strict = []
for ds in DATASETS:
    for model_name in MODELS_PER_DATASET:
        meta = conformal_meta_strict[f'{ds}/{model_name}']
        mthresh = meta['mondrian_thresholds_alpha_0.05']
        fb = set(meta['fallback_classes'])
        npc = meta['n_calib_per_predicted_class']
        for pred_cls in range(5):
            cs = str(pred_cls)
            t = mthresh[cs]
            n_calib = npc[cs]
            cliff = cliff_data_strict[(ds, model_name, pred_cls)]['cliff_fraction']
            s_t = flag_threshold(t)
            s_c = flag_cliff(cliff)
            s_s = flag_support(n_calib)
            overall = combine_flags(s_t, s_c, s_s)
            health_records_strict.append({
                'dataset': ds, 'model': model_name,
                'predicted_class_idx': pred_cls, 'predicted_class': CLASS_NAMES_5[pred_cls],
                'mondrian_threshold': t, 'is_fallback': pred_cls in fb,
                'n_calib': n_calib, 'cliff_fraction': cliff,
                'signal_threshold': s_t, 'signal_cliff': s_c, 'signal_support': s_s,
                'calib_health': overall,
            })

df_health_strict = pd.DataFrame(health_records_strict)
df_health_strict.to_csv(out_dir / 'scts_v2_calib_health_strict.csv', index=False)

print(f'Built strict health table: {len(df_health_strict)} rows')
print(f'\nClass-level (strict): {dict(df_health_strict["calib_health"].value_counts())}')
print(f'\nPer-dataset (strict):')
print(df_health_strict.groupby(['dataset', 'calib_health']).size().unstack(fill_value=0))

STRICT HEALTH FLAG
Built strict health table: 30 rows

Class-level (strict): {'red': np.int64(21), 'amber': np.int64(8), 'green': np.int64(1)}

Per-dataset (strict):
calib_health  amber  green  red
dataset                        
nsl_kdd_v2        8      1   21


In [13]:
# Augment per-sample SCTS with strict health flag
flag_lookup_strict = df_health_strict.set_index(['dataset', 'model', 'predicted_class_idx'])['calib_health'].to_dict()
df_scts_strict['calib_health'] = df_scts_strict.apply(
    lambda row: flag_lookup_strict.get((row['dataset'], row['model'], row['pred_class']), 'unknown'),
    axis=1,
)
df_scts_strict.to_csv(out_dir / 'scts_v2_canonical_with_health_strict.csv', index=False)

print(f'Augmented: {len(df_scts_strict)} per-sample rows')
print(f'Sample-level (strict): {dict(df_scts_strict["calib_health"].value_counts())}')

Augmented: 6000 per-sample rows
Sample-level (strict): {'red': np.int64(5033), 'amber': np.int64(811), 'green': np.int64(156)}


In [14]:
# Build strict health summary JSON
summary_by_flag_strict = {}
for flag in ['green', 'amber', 'red']:
    sub = df_scts_strict[df_scts_strict['calib_health'] == flag]
    if len(sub) == 0:
        summary_by_flag_strict[flag] = {'n_samples': 0}
        continue
    per_model_p = []
    for (ds, m), g in sub.groupby(['dataset', 'model']):
        if g['scts'].std() > 1e-9 and g['correct'].std() > 1e-9 and len(g) > 5:
            p = float(np.corrcoef(g['scts'], g['correct'])[0, 1])
            if not np.isnan(p):
                per_model_p.append(p)
    summary_by_flag_strict[flag] = {
        'n_samples': int(len(sub)),
        'mean_scts': float(sub['scts'].mean()),
        'mean_accuracy': float(sub['correct'].mean()),
        'n_models_valid_pearson': len(per_model_p),
        'mean_pearson_per_model': float(np.mean(per_model_p)) if per_model_p else None,
        'median_pearson_per_model': float(np.median(per_model_p)) if per_model_p else None,
        'min_pearson_per_model': float(np.min(per_model_p)) if per_model_p else None,
        'max_pearson_per_model': float(np.max(per_model_p)) if per_model_p else None,
    }

health_summary_strict = {
    'timestamp': datetime.now().isoformat(),
    'protocol': 'phase_a_strict_holdout_mondrian',
    'method': 'three_signal_health_flag_per_predicted_class',
    'signal_thresholds': {
        'threshold_green_range': [T_GREEN_LO, T_GREEN_HI],
        'threshold_red_high': T_RED_HI, 'threshold_red_low': T_RED_LO,
        'cliff_green_max': CLIFF_GREEN_HI, 'cliff_red_min': CLIFF_RED_LO,
        'cliff_score_threshold': CLIFF_THRESH,
        'support_green_min': N_GREEN_LO, 'support_red_max': N_RED_HI,
    },
    'overall_flag_counts': {
        'class_level': df_health_strict['calib_health'].value_counts().to_dict(),
        'sample_level': df_scts_strict['calib_health'].value_counts().to_dict(),
    },
    'per_dataset_class_level_counts': df_health_strict.groupby(['dataset', 'calib_health']).size().unstack(fill_value=0).to_dict(),
    'summary_by_flag': summary_by_flag_strict,
}

with open(out_dir / 'scts_v2_health_summary_strict.json', 'w') as f:
    json.dump(to_json_safe(health_summary_strict), f, indent=2)

print('Saved scts_v2_health_summary_strict.json')
print(f'\nPer-flag Pearson (strict):')
for f in ['green', 'amber', 'red']:
    s = summary_by_flag_strict[f]
    p = s.get('mean_pearson_per_model')
    p_str = f'{p:+.3f}' if p is not None else 'N/A'
    print(f'  {f.upper():>5}: n={s.get("n_samples", 0):>5}, mean_pearson={p_str}')

Saved scts_v2_health_summary_strict.json

Per-flag Pearson (strict):
  GREEN: n=  156, mean_pearson=+0.221
  AMBER: n=  811, mean_pearson=+0.199
    RED: n= 5033, mean_pearson=+0.041


In [15]:
# Bootstrap CIs on strict numbers (B=1000)
print('=' * 70)
print('BOOTSTRAP CIs ON STRICT PROTOCOL (B=1000)')
print('=' * 70)

rng_boot = np.random.RandomState(BOOTSTRAP_SEED)
bootstrap_records = []
t0 = time.time()

# 1/4: SCTS-correctness Pearson per (dataset, model)
print('1/4: SCTS-correctness Pearson per model...')
for (ds, m), g in df_scts_strict.groupby(['dataset', 'model']):
    s = g['scts'].values
    c = g['correct'].values
    if s.std() < 1e-9 or c.std() < 1e-9: continue
    n = len(s)
    boots = np.zeros(N_BOOTSTRAP, dtype=np.float32)
    for b in range(N_BOOTSTRAP):
        idx = rng_boot.randint(0, n, n)
        sb, cb = s[idx], c[idx]
        boots[b] = np.corrcoef(sb, cb)[0, 1] if sb.std() > 1e-9 and cb.std() > 1e-9 else np.nan
    boots = boots[~np.isnan(boots)]
    if len(boots) > 0:
        bootstrap_records.append({
            'protocol': 'strict', 'metric': 'scts_correctness_pearson',
            'dataset': ds, 'model': m,
            'point': float(np.corrcoef(s, c)[0, 1]),
            'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)),
            'n_bootstrap_valid': int(len(boots)),
        })

# 2/4: Per-flag Pearson means
print('2/4: Per-flag Pearson means...')
for flag in ['green', 'amber', 'red']:
    sub = df_scts_strict[df_scts_strict['calib_health'] == flag]
    per_m_p = []
    for (ds, m), g in sub.groupby(['dataset', 'model']):
        if g['scts'].std() > 1e-9 and g['correct'].std() > 1e-9 and len(g) > 5:
            p = float(np.corrcoef(g['scts'], g['correct'])[0, 1])
            if not np.isnan(p): per_m_p.append(p)
    if len(per_m_p) < 2: continue
    arr = np.array(per_m_p)
    n = len(arr)
    boots = np.array([arr[rng_boot.randint(0, n, n)].mean() for _ in range(N_BOOTSTRAP)])
    bootstrap_records.append({
        'protocol': 'strict', 'metric': 'per_flag_mean_pearson',
        'dataset': 'all', 'model': flag,
        'point': float(arr.mean()),
        'ci_low': float(np.percentile(boots, 2.5)),
        'ci_high': float(np.percentile(boots, 97.5)),
        'n_bootstrap_valid': N_BOOTSTRAP,
    })

# 3/4: GREEN-RED gap
print('3/4: GREEN-RED gap...')
g_p, r_p = [], []
for (ds, m), g in df_scts_strict[df_scts_strict['calib_health'] == 'green'].groupby(['dataset', 'model']):
    if g['scts'].std() > 1e-9 and g['correct'].std() > 1e-9 and len(g) > 5:
        v = float(np.corrcoef(g['scts'], g['correct'])[0, 1])
        if not np.isnan(v): g_p.append(v)
for (ds, m), g in df_scts_strict[df_scts_strict['calib_health'] == 'red'].groupby(['dataset', 'model']):
    if g['scts'].std() > 1e-9 and g['correct'].std() > 1e-9 and len(g) > 5:
        v = float(np.corrcoef(g['scts'], g['correct'])[0, 1])
        if not np.isnan(v): r_p.append(v)
if len(g_p) > 1 and len(r_p) > 1:
    ga, ra = np.array(g_p), np.array(r_p)
    boots = np.array([ga[rng_boot.randint(0, len(ga), len(ga))].mean() - ra[rng_boot.randint(0, len(ra), len(ra))].mean()
                      for _ in range(N_BOOTSTRAP)])
    bootstrap_records.append({
        'protocol': 'strict', 'metric': 'green_red_pearson_gap',
        'dataset': 'all', 'model': 'all',
        'point': float(ga.mean() - ra.mean()),
        'ci_low': float(np.percentile(boots, 2.5)),
        'ci_high': float(np.percentile(boots, 97.5)),
        'n_bootstrap_valid': N_BOOTSTRAP,
    })

# 4/4: R2L RED catch rate per dataset (with Wilson reference)
print('4/4: R2L RED catch rates per dataset...')
for ds in DATASETS:
    sub = df_scts_strict[(df_scts_strict['dataset'] == ds) & (df_scts_strict['true_class'] == 3)]
    if len(sub) == 0: continue
    is_red = (sub['calib_health'] == 'red').astype(int).values
    n = len(is_red)
    boots = np.array([is_red[rng_boot.randint(0, n, n)].mean() for _ in range(N_BOOTSTRAP)])
    k = int(is_red.sum())
    p_hat = k / n if n > 0 else 0
    z = 1.96
    denom = 1 + z**2/n
    centre = (p_hat + z**2/(2*n)) / denom
    halfw = z * np.sqrt(p_hat*(1-p_hat)/n + z**2/(4*n*n)) / denom
    bootstrap_records.append({
        'protocol': 'strict', 'metric': 'r2l_red_catch_rate',
        'dataset': ds, 'model': 'all',
        'point': float(p_hat),
        'ci_low': float(np.percentile(boots, 2.5)),
        'ci_high': float(np.percentile(boots, 97.5)),
        'wilson_ci_low': float(centre - halfw), 'wilson_ci_high': float(centre + halfw),
        'n_bootstrap_valid': N_BOOTSTRAP, 'n_total': int(n), 'n_red': int(k),
    })

df_boot_strict = pd.DataFrame(bootstrap_records)
df_boot_strict.to_csv(out_dir / 'bootstrap_cis_strict.csv', index=False)

print(f'\nBootstrap complete in {(time.time()-t0)/60:.1f} min, {len(bootstrap_records)} records')

# Show NSL R2L
nsl = df_boot_strict[(df_boot_strict['metric'] == 'r2l_red_catch_rate') & (df_boot_strict['dataset'] == 'nsl_kdd_v2')].iloc[0]
print(f'\nNSL R2L catch rate (strict):')
print(f'  {nsl["point"]*100:.1f}% bootstrap [{nsl["ci_low"]*100:.1f}%, {nsl["ci_high"]*100:.1f}%]')
print(f'  {nsl["point"]*100:.1f}% Wilson    [{nsl["wilson_ci_low"]*100:.1f}%, {nsl["wilson_ci_high"]*100:.1f}%]')
print(f'  v2 reference: 94.4% Wilson [93.0%, 95.5%]')

BOOTSTRAP CIs ON STRICT PROTOCOL (B=1000)
1/4: SCTS-correctness Pearson per model...
2/4: Per-flag Pearson means...
3/4: GREEN-RED gap...
4/4: R2L RED catch rates per dataset...

Bootstrap complete in 0.0 min, 9 records

NSL R2L catch rate (strict):
  79.4% bootstrap [77.3%, 81.5%]
  79.4% Wilson    [77.1%, 81.5%]
  v2 reference: 94.4% Wilson [93.0%, 95.5%]


In [16]:
# Build v2-vs-strict diff CSV
print('=' * 70)
print('BUILDING phase_a_v2_vs_strict_diff.csv')
print('=' * 70)

with open(out_dir / 'scts_v2_summary.json') as f:
    v2_summary = json.load(f)
df_v2_scts = pd.read_csv(out_dir / 'scts_v2_canonical.csv')
df_v2_health = pd.read_csv(out_dir / 'scts_v2_calib_health.csv')

diff_rows = []
for ds in DATASETS:
    for model_name in MODELS_PER_DATASET:
        key = f'{ds}/{model_name}'
        v2_meta = v2_summary['conformal_thresholds'][key]
        s_meta = conformal_meta_strict[key]

        v2_cell = df_v2_scts[(df_v2_scts['dataset'] == ds) & (df_v2_scts['model'] == model_name)]
        s_cell = df_scts_strict[(df_scts_strict['dataset'] == ds) & (df_scts_strict['model'] == model_name)]

        v2_p = float(np.corrcoef(v2_cell['scts'], v2_cell['correct'])[0, 1]) if v2_cell['scts'].std() > 1e-9 else 0.0
        s_p = float(np.corrcoef(s_cell['scts'], s_cell['correct'])[0, 1]) if s_cell['scts'].std() > 1e-9 else 0.0

        for pred_cls in range(5):
            cs = str(pred_cls)
            v2_t = v2_meta['mondrian_thresholds_alpha_0.05'][cs]
            s_t = s_meta['mondrian_thresholds_alpha_0.05'][cs]

            v2_flag_match = df_v2_health[(df_v2_health['dataset'] == ds) & (df_v2_health['model'] == model_name) & (df_v2_health['predicted_class_idx'] == pred_cls)]
            v2_flag = v2_flag_match['calib_health'].iloc[0] if len(v2_flag_match) > 0 else 'na'

            s_flag = df_health_strict[(df_health_strict['dataset'] == ds) & (df_health_strict['model'] == model_name) & (df_health_strict['predicted_class_idx'] == pred_cls)]['calib_health'].iloc[0]

            diff_rows.append({
                'dataset': ds, 'model': model_name,
                'predicted_class': CLASS_NAMES_5[pred_cls],
                'mondrian_thresh_v2': v2_t, 'mondrian_thresh_strict': s_t,
                'mondrian_thresh_delta': s_t - v2_t,
                'health_flag_v2': v2_flag, 'health_flag_strict': s_flag,
                'flag_changed': v2_flag != s_flag,
                'mean_scts_v2': float(v2_cell['scts'].mean()),
                'mean_scts_strict': float(s_cell['scts'].mean()),
                'mean_c3_v2': float(v2_cell['c3'].mean()),
                'mean_c3_strict': float(s_cell['c3'].mean()),
                'scts_pearson_v2': v2_p, 'scts_pearson_strict': s_p,
            })

df_diff = pd.DataFrame(diff_rows)
df_diff.to_csv(out_dir / 'phase_a_v2_vs_strict_diff.csv', index=False)

print(f'Saved phase_a_v2_vs_strict_diff.csv ({len(df_diff)} rows)')
print(f'\nFlag changes (v2 -> strict): {df_diff["flag_changed"].sum()} / {len(df_diff)}')
if df_diff['flag_changed'].sum() > 0:
    print(df_diff[df_diff['flag_changed']][['dataset', 'model', 'predicted_class', 'health_flag_v2', 'health_flag_strict']].to_string(index=False))

BUILDING phase_a_v2_vs_strict_diff.csv
Saved phase_a_v2_vs_strict_diff.csv (30 rows)

Flag changes (v2 -> strict): 14 / 30
   dataset            model predicted_class health_flag_v2 health_flag_strict
nsl_kdd_v2     rf_5class_cw             DoS          green                red
nsl_kdd_v2     rf_5class_cw             R2L          green              amber
nsl_kdd_v2    xgb_5class_cw             DoS          green                red
nsl_kdd_v2    xgb_5class_cw             R2L          green              amber
nsl_kdd_v2    dnn_5class_cw          Normal            red              amber
nsl_kdd_v2    dnn_5class_cw             DoS          green                red
nsl_kdd_v2    dnn_5class_cw           Probe          amber              green
nsl_kdd_v2    dnn_5class_cw             R2L          green              amber
nsl_kdd_v2  rf_5class_smote             DoS          green                red
nsl_kdd_v2 xgb_5class_smote             DoS          green                red
nsl_kdd_v2 xgb_5cla

In [17]:
# Headline comparison: v2 vs STRICT
print('=' * 70)
print('HEADLINE COMPARISON: v2 vs STRICT')
print('=' * 70)

print(f'\nOverall mean SCTS:')
print(f'  v2:     {v2_summary["overall_stats"]["mean_scts"]:.2f}')
print(f'  strict: {summary_strict["overall_stats"]["mean_scts"]:.2f}')

print(f'\nMean components (delta = strict - v2):')
for c in ['c1_calibration', 'c2_stability', 'c3_conformal']:
    vv, sv = v2_summary['mean_components'][c], summary_strict['mean_components'][c]
    print(f'  {c}: v2={vv:.3f}, strict={sv:.3f}, delta={sv-vv:+.3f}')

print(f'\nMean SCTS-correctness Pearson:')
print(f'  v2:     {v2_summary["mean_pearson_corr_scts_correctness"]:+.3f}')
print(f'  strict: {summary_strict["mean_pearson_corr_scts_correctness"]:+.3f}')

with open(out_dir / 'scts_v2_health_summary.json') as f:
    v2_h = json.load(f)

print(f'\nClass-level flag distribution:')
print(f'  v2:     {v2_h["overall_flag_counts"]["class_level"]}')
print(f'  strict: {health_summary_strict["overall_flag_counts"]["class_level"]}')

print(f'\nSample-level flag distribution:')
print(f'  v2:     {v2_h["overall_flag_counts"]["sample_level"]}')
print(f'  strict: {health_summary_strict["overall_flag_counts"]["sample_level"]}')

print(f'\nPer-flag mean Pearson:')
for f in ['green', 'amber', 'red']:
    vp = v2_h['summary_by_flag'][f].get('mean_pearson_per_model')
    sp = summary_by_flag_strict[f].get('mean_pearson_per_model')
    print(f'  {f.upper():>5}: v2={vp:+.3f}, strict={sp:+.3f}' if vp and sp else f'  {f.upper():>5}: v2={vp}, strict={sp}')

# NSL R2L catch rate
df_v2_merged = df_v2_scts[(df_v2_scts['dataset'] == 'nsl_kdd_v2') & (df_v2_scts['true_class'] == 3)].merge(
    df_v2_health[['dataset', 'model', 'predicted_class_idx', 'calib_health']],
    left_on=['dataset', 'model', 'pred_class'],
    right_on=['dataset', 'model', 'predicted_class_idx'],
)
n_v2 = len(df_v2_merged)
red_v2 = (df_v2_merged['calib_health'] == 'red').sum()

nsl_r2l_s = df_scts_strict[(df_scts_strict['dataset'] == 'nsl_kdd_v2') & (df_scts_strict['true_class'] == 3)]
n_s = len(nsl_r2l_s)
red_s = (nsl_r2l_s['calib_health'] == 'red').sum()

print(f'\n*** NSL R2L RED-flag catch rate (HEADLINE METRIC) ***')
print(f'  v2:     {red_v2}/{n_v2} = {100*red_v2/n_v2:.1f}%')
print(f'  strict: {red_s}/{n_s} = {100*red_s/n_s:.1f}%')

print('\n' + '=' * 70)
print('PHASE A COMPLETE')
print('=' * 70)

HEADLINE COMPARISON: v2 vs STRICT

Overall mean SCTS:
  v2:     78.69
  strict: 13.07

Mean components (delta = strict - v2):
  c1_calibration: v2=0.958, strict=0.958, delta=-0.000
  c2_stability: v2=0.562, strict=0.562, delta=+0.000
  c3_conformal: v2=0.945, strict=0.125, delta=-0.820

Mean SCTS-correctness Pearson:
  v2:     +0.086
  strict: +0.067

Class-level flag distribution:
  v2:     {'red': 18, 'green': 10, 'amber': 2}
  strict: {'red': 21, 'amber': 8, 'green': 1}

Sample-level flag distribution:
  v2:     {'red': 4392, 'green': 1443, 'amber': 165}
  strict: {'red': 5033, 'amber': 811, 'green': 156}

Per-flag mean Pearson:
  GREEN: v2=+0.095, strict=+0.221
  AMBER: v2=+0.245, strict=+0.199
    RED: v2=+0.173, strict=+0.041

*** NSL R2L RED-flag catch rate (HEADLINE METRIC) ***
  v2:     1216/1278 = 95.1%
  strict: 1015/1278 = 79.4%

PHASE A COMPLETE


In [ ]:
# Save mondrian_thresholds_strict.json
mondrian_export = {
    'protocol': 'phase_a_strict_holdout_mondrian',
    'timestamp': datetime.now().isoformat(),
    'thresholds_per_cell': conformal_meta_strict,
}
out_scts = Path(REPO) / 'results' / 'scts'
out_scts.mkdir(parents=True, exist_ok=True)
with open(out_scts / 'mondrian_thresholds_strict.json', 'w') as f:
    json.dump(to_json_safe(mondrian_export), f, indent=2)
print('Saved results/scts/mondrian_thresholds_strict.json')

Saved results/scts/mondrian_thresholds_strict.json


In [ ]:
# Commit and push (per work-unit discipline)
os.chdir(REPO)
!git status --short

print('\n>>> Staging files...')
!git add notebooks/07e_phase_a_strict_protocol.ipynb
!git add calibrators/*/X_calib_strict_indices.npy
!git add calibrators/*/X_mondrian_holdout_indices.npy
!git add calibrators/*/*_test_proba_strict.npy
!git add calibrators/*/*_holdout_proba_strict.npy
!git add calibrators/*/*_hybrid_strict.joblib
!git add results/scts/mondrian_thresholds_strict.json
!git add results/tables/scts_v2_canonical_strict.csv
!git add results/tables/scts_v2_canonical_with_health_strict.csv
!git add results/tables/scts_v2_per_class_summary_strict.csv
!git add results/tables/scts_v2_alpha_sensitivity_strict.csv
!git add results/tables/scts_v2_validation_strict.csv
!git add results/tables/scts_v2_summary_strict.json
!git add results/tables/scts_v2_calib_health_strict.csv
!git add results/tables/scts_v2_health_summary_strict.json
!git add results/tables/bootstrap_cis_strict.csv
!git add results/tables/phase_a_v2_vs_strict_diff.csv

!git status --short
!git commit -m "Phase A: strict Mondrian conformal (Option β) — class-aware 80/20 split of X_calib, refit calibrators on strict slice, fit Mondrian on disjoint holdout, recompute SCTS+health+bootstrap CIs, produce v2-vs-strict diff CSV"
!git push origin main

print('\n>>> Drive saved + git pushed? Verify status above shows clean tree before moving on.')

Refresh index: 100% (391/391), done.
 M notebooks/03_nsl_calibration_v2.ipynb
 M notebooks/03d_calibration_bootstrap_cis.ipynb
 M notebooks/03e_refit_hybrid_calibrators.ipynb
 M notebooks/04_shap_analysis.ipynb
 M notebooks/04b_calibration_shap_validation.ipynb
 M notebooks/04c_shap_canonical.ipynb
 M notebooks/05c_stability_canonical.ipynb
 M notebooks/06_krishna_agreement_v3.ipynb
 M notebooks/07d_scts_calib_health.ipynb
 M notebooks/08_bootstrap_cis.ipynb
 M results/tables/krishna_agreement_canonical_summary.json
?? calibrators/cic_ids2017_v2/dnn_5class_cw_hybrid_strict.joblib
?? calibrators/cic_ids2017_v2/dnn_5class_smote_hybrid_strict.joblib
?? calibrators/cic_ids2017_v2/rf_5class_cw_hybrid_strict.joblib
?? calibrators/cic_ids2017_v2/rf_5class_smote_hybrid_strict.joblib
?? calibrators/cic_ids2017_v2/xgb_5class_cw_hybrid_strict.joblib
?? calibrators/cic_ids2017_v2/xgb_5class_smote_hybrid_strict.joblib
?? calibrators/nsl_kdd_v2/dnn_5class_cw_hybrid_strict.joblib
?? calibrators/nsl_k